# 第 22 天：因子衰减

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：因子衰减
> 必做：半衰期分析
> 选做：滚动窗口
> 目标产出：衰减报告

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 理解不同持有期标签下 IC 的变化。
2. 学会估算半衰期，并用它指导调仓频率。
3. 用滚动窗口观察信号是否发生失效或风格切换。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

因子像一盏灯，有些亮得快也灭得快，有些不耀眼但能持续很久。半衰期分析就是看这盏灯的亮度多久会掉到一半。

## 5. 今日核心实验


### 实验 1：不同持有期下的 IC 衰减

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
target = factor_library["momentum_20"]
horizons = [1, 2, 5, 10, 20, 40, 60]
decay_rows = []

for h in horizons:
    label = close.shift(-h) / close - 1
    ic = rank_ic(target, label).dropna()
    decay_rows.append({
        "horizon": h,
        "ic_mean": ic.mean(),
        "ic_abs": abs(ic.mean()),
        "ic_ir": ic.mean() / ic.std() if ic.std() else np.nan,
        "positive_ratio": (ic > 0).mean(),
    })

decay_table = pd.DataFrame(decay_rows)
print(decay_table.round(4))


### 实验 2：估算半衰期：信号强度多久掉一半

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
valid = decay_table[decay_table["ic_abs"] > 1e-6].copy()
x = valid["horizon"].to_numpy(dtype=float)
y = np.log(valid["ic_abs"].to_numpy(dtype=float))
slope, intercept = np.polyfit(x, y, 1)
half_life = np.log(0.5) / slope if slope < 0 else np.inf

print({
    "log_decay_slope": round(float(slope), 6),
    "estimated_half_life_days": round(float(half_life), 2) if np.isfinite(half_life) else "未观察到衰减",
})


### 实验 3：画出衰减曲线

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(decay_table["horizon"], decay_table["ic_mean"], marker="o", label="IC mean")
ax.plot(decay_table["horizon"], decay_table["ic_abs"], marker="s", label="abs(IC)")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("因子 IC 随持有期衰减")
ax.set_xlabel("持有期")
ax.set_ylabel("IC")
ax.legend()
plt.tight_layout()
plt.show()
plt.close()


### 实验 4：滚动窗口：看信号是否阶段性失效

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
ic_5d = rank_ic(target, future_5d)
rolling_mean = ic_5d.rolling(60).mean()
rolling_ir = ic_5d.rolling(60).mean() / ic_5d.rolling(60).std()

regime = pd.DataFrame({
    "rolling_ic_mean": rolling_mean,
    "rolling_ic_ir": rolling_ir,
}).dropna()

print(regime.describe().round(4))


### 实验 5：把衰减结果翻译成调仓建议

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
turnover_by_horizon = {}
for h in [1, 5, 10, 20]:
    sampled_factor = target.iloc[::h]
    turnover_by_horizon[f"{h}日调仓"] = top_bucket_turnover(sampled_factor).mean()

suggestion = pd.Series(turnover_by_horizon).sort_index()
print("不同调仓频率的粗略换手压力：")
print(suggestion.round(4))


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：因子衰减
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：只看 5 日标签，误以为所有因子都适合 5 日交易。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：把负 IC 的绝对值当成好结果，却忘记检查方向。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：半衰期用样本内估计后直接交易，没有样本外验证。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：忽略交易成本，越短周期越容易被成本吃掉。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 23 天会进入因子拥挤和相关性，回答一批因子是不是太像了。

## 13. 一句话收尾

因子衰减 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须使用真实数据、真实成本、严格样本外检验和风险约束。
